# Phase 2 — Master Downstream Evaluation Runner (All 3 Tasks)

This notebook runs the complete Phase 2 Downstream Probe Evaluation Suite:
1. **Precompute Latents** for all 6 baseline models across all 5 stocks.
2. **Task 1: Trend Prediction Probing** (Macro-F1 & Accuracy).
3. **Task 2: Contiguous Imputation** (20-step Masked Test MSE & MAE).
4. **Task 3: Cross-Stock Transfer** (Source sz000001 $\rightarrow$ 4 Target Stocks, 20% Fine-Tuning Budget).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
!pip install -q lightning pandas numpy torch scikit-learn
print("✓ Environment and Google Drive ready.")


In [ ]:
# 1. Precompute Latents
!python cache_latents.py


In [ ]:
# 2. Run Task 1: Trend Prediction
import sys
from downstream_common import *
print("Running Trend Prediction Probe...")
%run TrendPrediction.ipynb


In [ ]:
# 3. Run Task 2: Contiguous Imputation
print("Running Contiguous Imputation Probe...")
%run Imputation.ipynb


In [ ]:
# 4. Run Task 3: Cross-Stock Transfer
print("Running Cross-Stock Transfer Probe...")
%run Transfer.ipynb


In [ ]:
# 5. Final Consolidated Summary
import pandas as pd
print("=" * 80)
print("  PHASE 2 CONSOLIDATED BENCHMARK SUMMARY")
print("=" * 80)
df_t1 = pd.read_csv("downstream_results/trend_prediction_results.csv")
df_t2 = pd.read_csv("downstream_results/imputation_results.csv")
df_t3 = pd.read_csv("downstream_results/transfer_results.csv")

t1_mean = df_t1.groupby('model')['macro_f1'].mean().rename('Task 1: Trend Macro-F1')
t2_mean = df_t2.groupby('model')['masked_test_mse'].mean().rename('Task 2: Impute MSE')
t3_mean = df_t3.groupby('model')['transfer_macro_f1'].mean().rename('Task 3: Transfer Macro-F1 (HEADLINE)')

df_summary = pd.concat([t1_mean, t2_mean, t3_mean], axis=1).sort_values(by='Task 3: Transfer Macro-F1 (HEADLINE)', ascending=False)
display(df_summary.round(4))
